# P1-4 · Học không giám sát và feature engineering

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q seaborn

## Lý thuyết

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# kiểm tra ví dụ 6 điểm ở trên, với đúng hai tâm khởi tạo kém 1 và 3
x6 = np.array([[1], [3], [5], [20], [22], [24]])
km6 = KMeans(n_clusters=2, init=np.array([[1.0], [3.0]]), n_init=1).fit(x6)
print(km6.labels_, km6.cluster_centers_.ravel(), km6.n_iter_)   # tâm 3 và 22

# 400 "khách hàng" giả lập có 4 nhóm, 2 đặc trưng đã chuẩn hóa
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
X, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.2, random_state=0)
X_scaled = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_scaled)
print(km.labels_[:12])                   # cụm của từng điểm
print(km.cluster_centers_.round(2))      # tọa độ tâm (trên thang đã chuẩn hóa)
print(round(km.inertia_, 1))

[0 0 0 1 1 1] [ 3. 22.] 3
[1 3 3 1 3 0 2 3 3 2 0 2]
[[-0.68  1.36]
 [ 0.39  0.09]
 [ 1.13 -1.15]
 [-1.11 -0.48]]
186.9


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
X_digits = load_digits().data             # 1797 ảnh 8×8 = 64 chiều
X_scaled = StandardScaler().fit_transform(X_digits)   # chuẩn hóa như trong hình ở trên
pca = PCA(n_components=0.95)              # giữ đủ thành phần để đạt 95% phương sai
Z = pca.fit_transform(X_scaled)
pca.n_components_, pca.explained_variance_ratio_[:3].round(3)

(40, array([0.12 , 0.096, 0.084]))


In [ ]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num = ["tuổi", "thu_nhập"]; cat = ["thành_phố", "kênh"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

# dữ liệu khách hàng giả lập, có ô trống
rng = np.random.default_rng(0); n = 800
df = pd.DataFrame({"tuổi": rng.integers(18, 65, n).astype(float), "thu_nhập": rng.gamma(2, 8, n).round(1),
                   "thành_phố": rng.choice(["HN", "HCM", "ĐN"], n), "kênh": rng.choice(["web", "app", "zalo"], n)})
y = ((df["thu_nhập"] > 15) & (df["kênh"] != "zalo")).astype(int) ^ (rng.random(n) < 0.1)
df.loc[rng.random(n) < 0.1, "thu_nhập"] = np.nan; df.loc[rng.random(n) < 0.05, "thành_phố"] = np.nan
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.25, random_state=0)

model.fit(X_train, y_train)                 # một lệnh, không rò rỉ
cross_val_score(model, X_train, y_train).round(3)    # mỗi fold tự fit lại toàn bộ tiền xử lý

array([0.875, 0.833, 0.858, 0.875, 0.858])


## Bài tập 4: Pipeline hoàn chỉnh {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

df = sns.load_dataset("titanic")
X, y = df[["age", "fare", "sex", "embarked", "class"]], df["survived"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ["age", "fare"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["sex", "embarked", "class"]),
])
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
print("Pipeline CV:", cross_val_score(pipe, X, y, cv=5).mean().round(4))
Xall = pre.fit_transform(X)   # sai: học trên toàn bộ dữ liệu trước khi CV
print("Rò rỉ CV:", cross_val_score(LogisticRegression(max_iter=1000), Xall, y, cv=5).mean().round(4))

Pipeline CV: 0.789
Rò rỉ CV: 0.789


## Bài tập 5: Đặt kho hàng bằng k-means {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
rng = np.random.default_rng(0)
khu = np.array([[3, 3], [12, 4], [6, 12], [14, 13]])
diem = np.vstack([c + rng.normal(0, 1.8, (100, 2)) for c in khu])
# TODO: km, kho (4 tâm), so_diem (số điểm mỗi kho), kc_tb

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(diem)
kho = km.cluster_centers_
so_diem = np.bincount(km.labels_)
kc_tb = np.linalg.norm(diem - kho[km.labels_], axis=1).mean()
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(diem[:, 0], diem[:, 1], c=km.labels_, s=8, alpha=0.6)
ax.scatter(kho[:, 0], kho[:, 1], marker="*", s=300, c="red", label="kho")
ax.set(title="4 kho đặt tại tâm cụm", xlabel="km", ylabel="km"); ax.legend()
print("vị trí kho:", kho.round(1).tolist(), "| số điểm:", so_diem, f"| khoảng cách TB {kc_tb:.2f} km")

vị trí kho: [[6.0, 12.2], [11.9, 3.9], [3.0, 3.2], [14.0, 12.9]] | số điểm: [102  97 103  98] | khoảng cách TB 2.24 km


## Bài tập 6: Bao nhiêu nhóm khách là hợp lý {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
X, _ = make_blobs(n_samples=600, centers=5, cluster_std=0.9, random_state=4)
# TODO: inertia, sil (list theo k = 2..8), k_tot

In [ ]:
ks = range(2, 9)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inertia.append(km.inertia_); sil.append(silhouette_score(X, km.labels_))
k_tot = ks[int(np.argmax(sil))]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(ks, inertia, marker="o"); axes[0].set(title="Elbow: inertia giảm chậm lại", xlabel="k")
axes[1].plot(ks, sil, marker="o"); axes[1].axvline(k_tot, color="gray", ls="--"); axes[1].set(title=f"Silhouette cao nhất ở k = {k_tot}", xlabel="k")
plt.tight_layout()
print([round(s, 3) for s in sil])

[0.545, 0.721, 0.771, 0.706, 0.645, 0.554, 0.454]


## Bài tập 7: Nén ảnh chữ số bằng PCA {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
X = load_digits().data          # (1797, 64), giá trị 0..16
# TODO: k90, Z (dữ liệu nén), X_rec (tái tạo), loi (MSE tái tạo)

In [ ]:
cum = np.cumsum(PCA().fit(X).explained_variance_ratio_)
k90 = int(np.argmax(cum >= 0.9) + 1)
pca = PCA(n_components=k90).fit(X)
Z = pca.transform(X)
X_rec = pca.inverse_transform(Z)
loi = np.mean((X - X_rec) ** 2)
fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for j in range(6):
    axes[0, j].imshow(X[j].reshape(8, 8), cmap="gray_r"); axes[0, j].axis("off")
    axes[1, j].imshow(X_rec[j].reshape(8, 8), cmap="gray_r"); axes[1, j].axis("off")
axes[0, 0].set_title("gốc (64 số)", loc="left"); axes[1, 0].set_title(f"tái tạo từ {k90} số", loc="left")
print(f"{k90} thành phần giữ {cum[k90 - 1]:.1%} phương sai; MSE tái tạo {loi:.2f}")

21 thành phần giữ 90.3% phương sai; MSE tái tạo 1.82


## Bài tập 8: Mã hóa đặc trưng đúng chỗ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
train = pd.DataFrame({"age": [25, 32, 47, 51, 23, 38], "income": [12, 20, 35, 40, 9, 22],
                      "city": ["Hà Nội", "TP.HCM", "Hà Nội", "Đà Nẵng", "TP.HCM", "Hà Nội"],
                      "channel": ["app", "web", "app", "store", "app", "web"]})
test = pd.DataFrame({"age": [29, 44], "income": [15, 30], "city": ["Cần Thơ", "TP.HCM"], "channel": ["web", "store"]})
# TODO: ct, Xtr, Xte

In [ ]:
ct = ColumnTransformer([
    ("num", StandardScaler(), ["age", "income"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["city", "channel"]),
])
Xtr = ct.fit_transform(train)
Xte = ct.transform(test)
print(ct.get_feature_names_out())
print(Xte.round(2))

['num__age' 'num__income' 'cat__city_Hà Nội' 'cat__city_TP.HCM'
 'cat__city_Đà Nẵng' 'cat__channel_app' 'cat__channel_store'
 'cat__channel_web']
[[-0.67 -0.71  0.    0.    0.    0.    0.    1.  ]
 [ 0.77  0.62  0.    1.    0.    0.    1.    0.  ]]
